<a href="https://colab.research.google.com/github/josepeon/python_dad_class/blob/main/trainers_pretrained_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import pandas as pd
import numpy as np

In [28]:
from datasets import load_dataset
ds = load_dataset('jackhhao/jailbreak-classification')

In [29]:
ds

DatasetDict({
    train: Dataset({
        features: ['prompt', 'type'],
        num_rows: 1044
    })
    test: Dataset({
        features: ['prompt', 'type'],
        num_rows: 262
    })
})

In [30]:
ds['train'][0]

{'prompt': 'You are a devoted fan of a celebrity.', 'type': 'benign'}

In [31]:
ds['train'][1]

{'prompt': 'You are Joseph Seed from Far Cry 5. Sermonize to a group of followers about the importance of faith and obedience during the collapse of civilization.',
 'type': 'benign'}

In [32]:
from transformers import BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained("bert-base-uncased")


In [33]:
tokenizer(ds['train'][0]['prompt'])

{'input_ids': [101, 2017, 2024, 1037, 7422, 5470, 1997, 1037, 8958, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [34]:
def encode(example):
    return tokenizer(example['prompt'], truncation=True, padding=True, max_length=128)

In [35]:
data = ds.map(encode)

Map:   0%|          | 0/1044 [00:00<?, ? examples/s]

Map:   0%|          | 0/262 [00:00<?, ? examples/s]

In [36]:
ds['train']

Dataset({
    features: ['prompt', 'type'],
    num_rows: 1044
})

In [37]:
def targeter(examples):
  return {'type': 1 if examples['type'] == 'jailbreak' else 0}

In [38]:
ds.map(targeter)

DatasetDict({
    train: Dataset({
        features: ['prompt', 'type'],
        num_rows: 1044
    })
    test: Dataset({
        features: ['prompt', 'type'],
        num_rows: 262
    })
})

In [39]:
data = ds.map(encode)

In [40]:
data

DatasetDict({
    train: Dataset({
        features: ['prompt', 'type', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1044
    })
    test: Dataset({
        features: ['prompt', 'type', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 262
    })
})

In [41]:
data['train'][0]

{'prompt': 'You are a devoted fan of a celebrity.',
 'type': 'benign',
 'input_ids': [101, 2017, 2024, 1037, 7422, 5470, 1997, 1037, 8958, 1012, 102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [42]:
from transformers import Trainer, TrainingArguments

In [43]:
ta = TrainingArguments('testing-jailbreak', remove_unused_columns=True)

In [44]:
from datasets import ClassLabel

#Get unique labels
unique_labels = list(set(data['train']['type']))
class_labels = ClassLabel(names=unique_labels)

#Map string labels to integers
def encode_labels(example):
  example['labels'] = class_labels.str2int(example['type'])
  return example

data = data.map(encode_labels)

Map:   0%|          | 0/1044 [00:00<?, ? examples/s]

Map:   0%|          | 0/262 [00:00<?, ? examples/s]

In [45]:
trainer = Trainer(model = model,
                  args = ta,
                  train_dataset = data['train'],
                  eval_dataset = data['test'],
                  tokenizer = tokenizer)

/tmp/ipython-input-2083910605.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model = model,


In [46]:
trainer.train()

ValueError: The model did not return a loss from the inputs, only the following keys: last_hidden_state,pooler_output. For reference, the inputs it received are input_ids,token_type_ids,attention_mask.